# resnet-stem — worked example 2: Stem downsamples any input by 4x

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `resnet-stem`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The ResNet stem's 4x spatial reduction is not specific to 224x224 — the two stride-2 stages each halve the spatial dimension, so any (sufficiently large) input is reduced by 4x. The channel dimension always becomes 64 regardless of input size.

## Worked solution

We build the same stem and feed it a non-standard `(2, 3, 96, 128)` input. Each stride-2 stage halves H and W: conv gives `(96+6-7)//2+1 = 48` and `(128+6-7)//2+1 = 64`; maxpool gives `(48+2-3)//2+1 = 24` and `(64+2-3)//2+1 = 32`. So the output is `(2, 64, 24, 32)` — exactly H/4 and W/4. We print the output shape to show the downsample factor generalizes and channels are lifted to 64 for a batch of 2.

In [ ]:
import torch.nn as nn


def build_stem():
    return nn.Sequential(
        nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
        nn.BatchNorm2d(64),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
    )


stem = build_stem().eval()
x = t.zeros(2, 3, 96, 128)
out = stem(x)
print('output shape:', tuple(out.shape))
print('H reduced 4x:', 96 // out.shape[2] == 4)
print('W reduced 4x:', 128 // out.shape[3] == 4)